In [24]:
import pandas as pd
import logging
import os

#configure the logging

logging.basicConfig(
    level = logging.INFO,
    format = '%(asctime)s - %(levelname)s - %(message)s',
    handlers = [
        logging.FileHandler('SalesPipeline_run.log'),
        logging.StreamHandler()    
    ]
)

logger = logging.getLogger(__name__)

class SalesPipeline :
    def __init__(self, config):
        self.filepath   = config["filepath"]        # required — no default
        self.fill_value = config.get("fill_value", 0)
        self.drop_col   = config.get("drop_col", None)
        self.group_col  = config["group_col"]       # required — no default
        self.value_col  = config["value_col"]       # required — no default
        self.agg_func   = config.get("agg_func", "sum")
        self.output_dir = config.get("output_dir", "output/")
        self.df         = None
        self.result     = None

    def read(self) :
        try :
             self.df = pd.read_csv(self.filepath)
             logger.info(f"loaded {self.filepath} - {self.df.shape}")
             return self
        except FileNotFoundError :
            logger.error(f"file not found : {self.filepath}")
            return None
        except UnicodeDecodeError :
            logger.error(f"encoding error - try encoding = 'Latin-1'")
            return None
        except Exception as e:
            logger.error(f"Unexpected error : {e}")
            return None

    def clean(self) :
        try :
            self.df = self.df.fillna(self.fill_value)
            if self.drop_col :
                self.df = self.df.drop(columns = self.drop_col)
            self.df = self.df.reset_index(drop = True)
            logger.info(f"Cleaned — {self.df.shape}")
            return self
        
        except KeyError as e :
            logger.error(f"column not found: {e}")
            return None
        except Exception as e :
            logger.error(f"cleaning failed: {e}")
            return None
            
    def transform(self) :
        try :
            self.result = self.df.groupby(self.group_col)[self.value_col].agg(self.agg_func).reset_index()
            self.result = self.result.rename(columns={self.value_col: f'{self.agg_func}_{self.value_col}'})
            logger.info(f"transformation completed: {self.result.shape}")
            return self
        except Exception as e :
            logger.error(f"transformation failed :{e}")
            return None
            
    def save(self) :
        try:
            os.makedirs(self.output_dir,exist_ok = True)
            self.result.to_csv(self.output_dir + "sales_report.csv", index=False)
            logger.info(f"CSV written: {self.output_dir}sales_report.csv")
            self.result.to_parquet(self.output_dir + "sales_report.parquet", engine='pyarrow', index=False)
            logger.info(f"Parquet written: {self.output_dir}sales_report.parquet")
            return self
        except Exception as e :
            logger.error(f"output file creation failed : {e}")
            return None
        
    def run(self) :
        self.read()
        if self.df is None :
            logger.error(f"error while reading the into the dataframe. Check if the \"{self.filepath}\" file is available")
            return None
                
        self.clean()
        if self.df is None :
            logger.error(f"error occured while cleaning the dataframe")
            return None
            
                
        self.transform()
        if self.result is None :
            logger.error(f"error occured while transforming the dataframe")
            return None
            
                
        result = self.save()
        if result is None:
            logger.error("Pipeline stopped - save failed")
            return None
        logger.info("Pipeline completed successfully")
        return self.result
            
                
                
                    
        
    def __repr__(self):
           return f"Pipeline(filepath={self.filepath}, fill_value={self.fill_value}, drop_col={self.drop_col})"    
           
             

        
        
config = {
    "filepath"  : "sales_data.csv",
    "fill_value": 0,
    "drop_col"  : None,
    "group_col" : "region",
    "value_col" : "sales",
    "agg_func"  : "sum",
    "output_dir": "output/"
}

p1 = SalesPipeline(config)
result = p1.run()
print(result)

2026-08-14 12:15:30,259 - INFO - loaded sales_data.csv - (10, 9)
2026-08-14 12:15:30,265 - INFO - Cleaned — (10, 9)
2026-08-14 12:15:30,300 - INFO - transformation completed: (3, 2)
2026-08-14 12:15:30,308 - INFO - CSV written: output/sales_report.csv
2026-08-14 12:15:30,316 - INFO - Parquet written: output/sales_report.parquet
2026-08-14 12:15:30,318 - INFO - Pipeline completed successfully


  region  sum_sales
0  North      56000
1  South      40000
2   West      60000
